In [ ]:
import os
import pympi
import pandas as pd
import torch, gc
import ast

from diarizers import SegmentationModel
from pyannote.audio import Pipeline, Audio
from pyannote.core import Segment, Annotation
from pyannote.metrics.diarization import DiarizationErrorRate

torch.cuda.empty_cache()
gc.collect()

device = torch.device("cuda")

# Charger le pipeline pyannote
pipeline = Pipeline.from_pretrained(
    'pyannote/speaker-diarization-3.1',
    use_auth_token=""               ################################## AJOUTER TOKEN HUGGINGFACE AVEC ACCES MODELES PYANNOTE
).to(device)

# Charger le modèle de segmentation custom
model = SegmentationModel().from_pretrained("Rziane/speaker-segmentation-ESLO-CAENNAIS28.04.25")
model = model.to_pyannote_model()
pipeline._segmentation.model = model.to(device)

num_speakers = 3

# Affichage des hyperparamètres
default_hyperparameters = pipeline.parameters(instantiated=True)
for param, value in default_hyperparameters.items():
    print(f"{param}: {value}")

# Chargement d’un fichier ELAN
def load_elan_annotations(elan_file):
    eaf = pympi.Elan.Eaf(elan_file)
    annotations = {}
    for tier in eaf.get_tier_names():
        if "EXTRA" in tier:
            continue
        speaker_annotations = []
        for start_time, end_time, _ in eaf.get_annotation_data_for_tier(tier):
            speaker_annotations.append(Segment(start_time / 1000, end_time / 1000))
        annotations[tier] = speaker_annotations
    return annotations

# Chargement d’un fichier TSV
def load_tsv_annotations(tsv_file):
    df = pd.read_csv(tsv_file, sep='\t')
    annotations = {}

    for _, row in df.iterrows():
        speaker = row['speakers']
        start_times = ast.literal_eval(row['timestamps_start'])
        end_times = ast.literal_eval(row['timestamps_end'])

        if speaker not in annotations:
            annotations[speaker] = []

        for start, end in zip(start_times, end_times):
            annotations[speaker].append(Segment(float(start), float(end)))

    return annotations

# Évaluer et extraire les composantes détaillées de DER
def evaluate_metrics(reference, hypothesis):
    metric = DiarizationErrorRate()
    detailed = metric(reference, hypothesis, detailed=True)

    der = detailed["diarization error rate"]
    miss = detailed["missed detection"] / detailed["total"]
    fa = detailed["false alarm"] / detailed["total"]
    conf = detailed["confusion"] / detailed["total"]

    return {
        "DER": der,
        "Missed detection": miss,
        "False alarm": fa,
        "Confusion": conf
    }

# Évaluation principale
def evaluate_pyannote(annotation_file, audio_file=None):
    if annotation_file.endswith('.eaf'):
        annotations = load_elan_annotations(annotation_file)

        io = Audio(mono='downmix', sample_rate=16000)
        waveform, sample_rate = io(audio_file)
        diarization = pipeline({"waveform": waveform, "sample_rate": sample_rate}, num_speakers=num_speakers)

        reference = Annotation()
        for speaker, segments in annotations.items():
            for segment in segments:
                reference[segment] = speaker

        scores = evaluate_metrics(reference, diarization)
        for name, val in scores.items():
            print(f"{name} = {100 * val:.2f}%")
        return scores

    elif annotation_file.endswith('.tsv'):
        df = pd.read_csv(annotation_file, sep='\t')
        if 'audio' not in df.columns:
            raise ValueError("Le fichier TSV doit contenir une colonne 'audio'.")

        results = []

        for audio_path, group in df.groupby("audio"):
            print(f"\nTraitement de : {audio_path}")
            annotations = {}

            for _, row in group.iterrows():
                speaker = row['speakers']
                start_times = ast.literal_eval(row['timestamps_start'])
                end_times = ast.literal_eval(row['timestamps_end'])

                if speaker not in annotations:
                    annotations[speaker] = []

                for start, end in zip(start_times, end_times):
                    annotations[speaker].append(Segment(float(start), float(end)))

            io = Audio(mono='downmix', sample_rate=16000)
            waveform, sample_rate = io(audio_path)
            diarization = pipeline({"waveform": waveform, "sample_rate": sample_rate}, num_speakers=num_speakers)

            reference = Annotation()
            for speaker, segments in annotations.items():
                for segment in segments:
                    reference[segment] = speaker

            scores = evaluate_metrics(reference, diarization)
            scores["audio"] = audio_path
            print("\n".join([f"{k} = {100*v:.2f}%" for k, v in scores.items() if k != "audio"]))
            results.append(scores)

        # Moyennes
        metrics = ["DER", "Missed detection", "False alarm", "Confusion"]
        avg = {k: sum(d[k] for d in results) / len(results) for k in metrics}
        print("\nMoyennes globales :")
        for k, v in avg.items():
            print(f"{k} = {100 * v:.2f}%")

        return results, avg

    else:
        raise ValueError("Format de fichier non supporté. Utilise .tsv ou .eaf")


segmentation: {'min_duration_off': 0.0}
clustering: {'method': 'centroid', 'min_cluster_size': 12, 'threshold': 0.7045654963945799}


In [11]:
# Exemple d'appel avec fichier ELAN ou TSV
annotation_file = "/mnt/PROJET_CAENNAIS/commun/datasets_expe_frapeor_diarisation/datasets_v3/test/GC_E2_train.tsv"  # ou "/mnt/data/GC_E2_train.tsv"
audio_file = "/mnt/PROJET_CAENNAIS/corpus/GC/1_enregistrements/wav/GC_E2_splits/GC_E2_part1.wav"      # obligatoire si .eaf
evaluate_pyannote(annotation_file, audio_file)


Traitement de : /mnt/PROJET_CAENNAIS/commun/datasets_expe_frapeor_diarisation/audio/cropped_audio_v3/GC_E2_part1.wav


/home/ziane212/anaconda3/envs/finetuning_pyannote2/lib/python3.11/site-packages/pyannote/audio/models/blocks/pooling.py:104: UserWarning: std(): degrees of freedom is <= 0. Correction should be strictly less than the reduction factor (input numel divided by output numel). (Triggered internally at ../aten/src/ATen/native/ReduceOps.cpp:1808.)
  std = sequences.std(dim=-1, correction=1)
/home/ziane212/anaconda3/envs/finetuning_pyannote2/lib/python3.11/site-packages/pyannote/metrics/utils.py:200: UserWarning: 'uem' was approximated by the union of 'reference' and 'hypothesis' extents.
  warnings.warn(


DER = 75.45% → Missed detection = 20.87% → False alarm = 8.05% → Confusion = 46.53%

Traitement de : /mnt/PROJET_CAENNAIS/commun/datasets_expe_frapeor_diarisation/audio/cropped_audio_v3/GC_E2_part2.wav


/home/ziane212/anaconda3/envs/finetuning_pyannote2/lib/python3.11/site-packages/pyannote/metrics/utils.py:200: UserWarning: 'uem' was approximated by the union of 'reference' and 'hypothesis' extents.
  warnings.warn(


DER = 67.33% → Missed detection = 23.01% → False alarm = 4.02% → Confusion = 40.30%

Traitement de : /mnt/PROJET_CAENNAIS/commun/datasets_expe_frapeor_diarisation/audio/cropped_audio_v3/GC_E2_part3.wav


...

DER = 55.08% → Missed detection = 18.08% → False alarm = 5.79% → Confusion = 31.21%

Traitement de : /mnt/PROJET_CAENNAIS/commun/datasets_expe_frapeor_diarisation/audio/cropped_audio_v3/GC_E2_part4.wav


/home/ziane212/anaconda3/envs/finetuning_pyannote2/lib/python3.11/site-packages/pyannote/metrics/utils.py:200: UserWarning: 'uem' was approximated by the union of 'reference' and 'hypothesis' extents.
  warnings.warn(


DER = 49.54% → Missed detection = 20.49% → False alarm = 5.70% → Confusion = 23.35%

Traitement de : /mnt/PROJET_CAENNAIS/commun/datasets_expe_frapeor_diarisation/audio/cropped_audio_v3/GC_E2_part5.wav
DER = 69.27% → Missed detection = 19.82% → False alarm = 5.58% → Confusion = 43.87%

Moyennes globales :
DER = 63.33%
Missed detection = 20.45%
False alarm = 5.83%
Confusion = 37.05%


/home/ziane212/anaconda3/envs/finetuning_pyannote2/lib/python3.11/site-packages/pyannote/metrics/utils.py:200: UserWarning: 'uem' was approximated by the union of 'reference' and 'hypothesis' extents.
  warnings.warn(


([{'DER': 0.7544514401050413,
   'Missed detection': 0.20865168827057048,
   'False alarm': 0.08046697489477543,
   'Confusion': 0.4653327769396953,
   'audio': '/mnt/PROJET_CAENNAIS/commun/datasets_expe_frapeor_diarisation/audio/cropped_audio_v3/GC_E2_part1.wav'},
  {'DER': 0.6732958282065434,
   'Missed detection': 0.23008122427765154,
   'False alarm': 0.040215829055115865,
   'Confusion': 0.4029987748737761,
   'audio': '/mnt/PROJET_CAENNAIS/commun/datasets_expe_frapeor_diarisation/audio/cropped_audio_v3/GC_E2_part2.wav'},
  {'DER': 0.5507893683574673,
   'Missed detection': 0.18083159437676458,
   'False alarm': 0.05789933342344819,
   'Confusion': 0.31205844055725435,
   'audio': '/mnt/PROJET_CAENNAIS/commun/datasets_expe_frapeor_diarisation/audio/cropped_audio_v3/GC_E2_part3.wav'},
  {'DER': 0.49543617457802863,
   'Missed detection': 0.2048782367447598,
   'False alarm': 0.0570270841329075,
   'Confusion': 0.2335308537003613,
   'audio': '/mnt/PROJET_CAENNAIS/commun/datasets_ex

In [ ]:
B1
pyannote
Diarization Error Rate (DER) = 34.00%

simsamu
Diarization Error Rate (DER) = 37.87%

CAENNAIS
Diarization Error Rate (DER) = 30.80%

B2
pyannote
Diarization Error Rate (DER) = 25.68%

simsamu
Diarization Error Rate (DER) = 31.13%

CAENNAIS
Diarization Error Rate (DER) = 23.84%

B3
pyannote
Diarization Error Rate (DER) = 12.08%

simsamu
Diarization Error Rate (DER) = 20.04%

CAENNAIS
Diarization Error Rate (DER) = 26.82%

C1
pyannote
Diarization Error Rate (DER) = 145.13%

simsamu
Diarization Error Rate (DER) = 165.41%

CAENNAIS
Diarization Error Rate (DER) = 100.03%

C2 (23min)
pyannote
Diarization Error Rate (DER) = 84.90%

simsamu
Diarization Error Rate (DER) = 89.11%

CAENNAIS
Diarization Error Rate (DER) = 93.97%

CAENNAIS v2
Diarization Error Rate (DER) = 58.39%